# DS-03 – Détection des clients à risque d'attrition
TuniShop – dossier TS-DATA-DSC-001 – v1.0 – 14/10/2026

**Question** : identifier les clients actifs qui ne commanderont plus dans les 6 prochains mois (BM-07, PR-3), et vérifier si les mauvaises expériences augmentent ce risque (Q7).

**Définitions** : client actif = au moins une commande valide dans les 12 mois précédant la date d'observation ; attrition = aucune commande valide dans les 6 mois suivants. Apprentissage : observation au 01/07/2024 ; test : observation au 01/07/2025 (fin des données au 31/12/2025).

In [1]:
import os, json, warnings
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
# Dossier donnees_reference du jeu TuniShop (variable d'environnement ou chemin relatif)
R = os.environ.get("TUNISHOP_REF_DIR", "../donnees_reference") + "/"
FIG = os.environ.get("TUNISHOP_FIG_DIR", "figures") + "/"; os.makedirs(FIG, exist_ok=True)
SEED = 42
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, roc_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

## 1. Données et mauvaises expériences
Une mauvaise expérience : retard de livraison d'au moins 3 jours, retour, note ≤ 2, ou annulation pour rupture.

In [2]:
c=pd.read_csv(R+'commandes.csv',parse_dates=['date_commande']); l=pd.read_csv(R+'livraisons.csv',parse_dates=['date_livraison_prevue','date_livraison_reelle'])
r=pd.read_csv(R+'remboursements.csv'); a=pd.read_csv(R+'avis_clients.csv'); cl=pd.read_csv(R+'clients.csv',parse_dates=['date_inscription'])
v=c[c.statut_commande!='paiement_echoue'].copy()
l['retard3']=(l.date_livraison_reelle.dt.normalize()-l.date_livraison_prevue).dt.days>=3
l['refus']=l.statut_livraison=='echec_retour_expediteur'
v=v.merge(l[['commande_id','retard3','refus']],on='commande_id',how='left')
ret=r[r.type_remboursement=='retour'].groupby('commande_id').size().rename('n_ret')
v=v.merge(ret,on='commande_id',how='left'); v['n_ret']=v.n_ret.fillna(0)
v=v.merge(a.groupby('commande_id').note.min().rename('note_min'),on='commande_id',how='left')
v['mauvaise']=(v.retard3.fillna(False))|(v.n_ret>0)|(v.note_min<=2)|(v.statut_commande=='annulee')&(v.motif_annulation=='rupture_stock')


## 2. Photographies des clients actifs
Toutes les variables sont calculées avec les données antérieures à la date d'observation. Le segment de fidélité du fichier clients est **exclu** : il est calculé sur le chiffre d'affaires 2025, donc postérieur à la date d'observation (fuite d'information).

In [3]:
def snap(t0):
    t0=pd.Timestamp(t0); h=v[(v.date_commande<t0)&(v.date_commande>=t0-pd.DateOffset(months=12))]; f=v[(v.date_commande>=t0)&(v.date_commande<t0+pd.DateOffset(months=6))]
    hist=v[v.date_commande<t0]
    X=h.groupby('client_id').agg(n12=('commande_id','size'),ca12=('montant_total_ttc','sum'),panier=('montant_total_ttc','mean'),der=('date_commande','max'),
        mauv=('mauvaise','sum'),retards=('retard3','sum'),refus=('refus','sum'),retours=('n_ret','sum'),app=('canal',lambda x:(x=='application_mobile').mean()),cod=('mode_paiement',lambda x:(x=='paiement_a_la_livraison').mean()))
    X['recence_j']=(t0-X.der).dt.days; X=X.drop(columns='der')
    X['n_total']=hist.groupby('client_id').size().reindex(X.index)
    X=X.join(cl.set_index('client_id')[['type_client','canal_inscription','date_inscription','opt_in_email']])  # segment_fidelite exclu : calculé sur le CA 2025, donc postérieur à l'observation (fuite)
    X['anciennete_j']=(t0-X.date_inscription).dt.days; X=X.drop(columns=['date_inscription'])
    X['taux_mauv']=X.mauv/X.n12
    X['y']=(~X.index.isin(f.client_id)).astype(int)
    for col in ['retards','refus']: X[col]=X[col].astype(float)
    return X
tr=snap('2024-07-01'); te=snap('2025-07-01')
pd.DataFrame({'clients':[len(tr),len(te)],'taux_attrition':[tr.y.mean(),te.y.mean()]},index=['observation 01/07/2024','observation 01/07/2025']).round(3)

,clients,taux_attrition
observation 01/07/2024,4725,0.566
observation 01/07/2025,6322,0.559


## 3. Modèles et référence
Référence PR-3 : seuil d'inactivité de 90 jours. Le critère compare la précision des modèles **au même rappel** que ce seuil.

In [4]:
num=['n12','ca12','panier','mauv','retards','refus','retours','app','cod','recence_j','n_total','anciennete_j','taux_mauv']; cat=['type_client','canal_inscription','opt_in_email']
for x in cat: tr[x]=tr[x].astype(str); te[x]=te[x].astype(str)
res={'Seuil : inactif depuis plus de 90 jours':(te.recence_j>90).astype(int).values, 'Récence seule (score)':te.recence_j.values}
lr=make_pipeline(ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(handle_unknown='ignore'),cat)]),LogisticRegression(max_iter=3000)); lr.fit(tr[num+cat],tr.y); res['Régression logistique']=lr.predict_proba(te[num+cat])[:,1]
gb=make_pipeline(ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat)]),HistGradientBoostingClassifier(max_iter=300,learning_rate=0.05,max_leaf_nodes=15,min_samples_leaf=40,random_state=42)); gb.fit(tr[num+cat],tr.y); res['Gradient boosting']=gb.predict_proba(te[num+cat])[:,1]


In [5]:
yb=(te.recence_j>90).astype(int); rb=recall_score(te.y,yb); pb=precision_score(te.y,yb); y=te.y.values; lignes=[]
for n,sc in res.items():
    o=np.argsort(-np.asarray(sc,dtype=float)); k=int(round(rb*y.sum())); cum=np.cumsum(y[o]); idx=int(np.searchsorted(cum,k)); prec=k/(idx+1)
    lignes.append(dict(modele=n,AUC=roc_auc_score(y,sc),AP=average_precision_score(y,sc),precision_au_rappel_du_seuil=prec))
resultats=pd.DataFrame(lignes).round(3); print('Seuil 90 jours : précision',round(pb,3),'rappel',round(rb,3)); resultats

Seuil 90 jours : précision 0.769 rappel 0.755


,modele,AUC,AP,precision_au_rappel_du_seuil
0,Seuil : inactif depuis plus de 90 jours,0.734,0.718,0.769
1,Récence seule (score),0.810,0.850,0.769
2,Régression logistique,0.823,0.858,0.778
3,Gradient boosting,0.818,0.853,0.769


In [6]:
fig,ax=plt.subplots(figsize=(6,5))
for n,sc in res.items():
    f,t,_=roc_curve(y,sc); ax.plot(f,t,label=f"{n} (AUC {roc_auc_score(y,sc):.3f})")
ax.plot([0,1],[0,1],'k--',lw=0.8); ax.set_xlabel('Taux de faux positifs'); ax.set_ylabel('Taux de vrais positifs'); ax.set_title('Attrition – courbes ROC (observation 01/07/2025)'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG+'ds03_roc.png',dpi=150)

## 4. Variables les plus influentes (régression logistique, coefficients standardisés)

In [7]:
noms=lr[0].get_feature_names_out(); coef=pd.Series(lr[-1].coef_[0],index=noms).sort_values(key=abs,ascending=False).head(10).round(3); coef

n__recence_j                            1.514
n__n12                                 -0.441
c__type_client_particulier              0.306
n__refus                                0.226
n__retards                              0.186
c__type_client_professionnel           -0.172
n__n_total                              0.123
c__opt_in_email_False                   0.103
c__canal_inscription_reseaux_sociaux    0.085
n__retours                             -0.053
dtype: float64

## 5. Effet des mauvaises expériences, à nombre de commandes égal

In [8]:
te2=te.copy(); te2['a_mauvaise_experience']=te2.mauv>0; te2['commandes_12_mois']=pd.cut(te2.n12,[0,1,2,4,100],labels=['1','2','3-4','5 et plus'])
effet=te2.groupby(['commandes_12_mois','a_mauvaise_experience']).y.mean().unstack().round(3); effet['clients']=te2.groupby('commandes_12_mois').size(); effet

a_mauvaise_experience,False,True,clients
commandes_12_mois,,,
1,0.707,0.706,1987
2,0.622,0.621,1228
3-4,0.539,0.566,1314
5 et plus,0.347,0.360,1793


## 6. Conclusion

In [9]:
r=resultats.set_index('modele'); seuil=r.loc['Seuil : inactif depuis plus de 90 jours','precision_au_rappel_du_seuil']; meilleur=r.drop(index=['Seuil : inactif depuis plus de 90 jours','Récence seule (score)']).precision_au_rappel_du_seuil.idxmax()
succes=bool(r.loc[meilleur,'precision_au_rappel_du_seuil']>seuil)
json.dump(dict(resultats=resultats.to_dict('records'),seuil=dict(precision=pb,rappel=rb),coef=coef.to_dict(),effet=effet.reset_index().astype(str).to_dict('records'),succes=succes,meilleur=meilleur,n_train=len(tr),n_test=len(te),taux_test=float(te.y.mean())),open(FIG+'ds03_resultats.json','w'),ensure_ascii=False,default=float)
print('Meilleur modèle :',meilleur,'| critère PR-3 atteint :',succes)

Meilleur modèle : Régression logistique | critère PR-3 atteint : True
